In [1]:
# imports
import pandas as pd
import numpy as np

In [2]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)
library_path

PosixPath('/mnt/DataVol/Beratungen/Yurttas/survival/src')

In [3]:
# load data
DATA_PATH = library_path.parent / "data"

data_df = pd.read_csv(f"{DATA_PATH}/Excel table survival analysis.csv", sep="\t", encoding="utf-8")
data_df.head()

,Bday,OPDate,Sex,Zn HIPEC,Age,Tumor,Histo,T,N,M,...,CC,Tod Datum,Overall survival,Overall survival (months),Rezidiv,Lok Rezidiv,Datum Rezidiv,Recurrence-free survival,Recurrence-free survival (months),SF Grund
0,2/25/1991,3/10/2008,2,1,17,1,3,4,2,1,...,1,8/16/2008,"0 years, 5 months, 6 days",5.20,3,1,8/12/2008,"0 years, 5 months, 2 days",5.07,0
1,7/17/1993,7/14/2015,1,1,21,1,3,4,1,1,...,0,5/6/2018,"2 years, 9 months, 22 days",33.72,0,1,12/22/2016,"1 years, 5 months, 8 days",17.26,0
2,6/8/1995,11/23/2018,2,1,23,4,2,3,1,1,...,1,99,99,NaN,3,1,6/22/2022,"3 years, 6 months, 30 days",42.99,0
3,9/2/1979,11/15/2005,1,1,26,10,1,99,99,1,...,0,5/11/2009,"3 years, 5 months, 26 days",41.85,0,1.8,4/12/2006,"0 years, 4 months, 28 days",4.92,0
4,11/18/1990,12/29/2017,1,1,27,3,3,4,2,1,...,0,11/3/2019,"1 years, 10 months, 5 days",22.16,0,1,7/19/2018,"0 years, 6 months, 20 days",6.66,0


In [4]:
# selecting columns to use for analysis
cols_to_use = [
    "Bday", "OPDate", "Sex", "Age", "Tumor","sPCI", "pPCI", "Tod Datum", "Datum Rezidiv", "CC"
]

reduc_df = data_df[cols_to_use].copy()
reduc_df.head()

,Bday,OPDate,Sex,Age,Tumor,sPCI,pPCI,Tod Datum,Datum Rezidiv,CC
0,2/25/1991,3/10/2008,2,17,1,26,22,8/16/2008,8/12/2008,1
1,7/17/1993,7/14/2015,1,21,1,5,3,5/6/2018,12/22/2016,0
2,6/8/1995,11/23/2018,2,23,4,15,6,99,6/22/2022,1
3,9/2/1979,11/15/2005,1,26,10,15,6,5/11/2009,4/12/2006,0
4,11/18/1990,12/29/2017,1,27,3,18,4,11/3/2019,7/19/2018,0


In [5]:
# converting date columns to datetime format
reduc_df["Bday"] = pd.to_datetime(reduc_df["Bday"], errors="coerce")
reduc_df["OPDate"] = pd.to_datetime(reduc_df["OPDate"], errors="coerce")
reduc_df["Tod Datum"] = pd.to_datetime(reduc_df["Tod Datum"], errors="coerce")
reduc_df["Datum Rezidiv"] = pd.to_datetime(reduc_df["Datum Rezidiv"], errors="coerce")
reduc_df.head()

,Bday,OPDate,Sex,Age,Tumor,sPCI,pPCI,Tod Datum,Datum Rezidiv,CC
0,1991-02-25,2008-03-10,2,17,1,26,22,2008-08-16,2008-08-12,1
1,1993-07-17,2015-07-14,1,21,1,5,3,2018-05-06,2016-12-22,0
2,1995-06-08,2018-11-23,2,23,4,15,6,NaT,2022-06-22,1
3,1979-09-02,2005-11-15,1,26,10,15,6,2009-05-11,2006-04-12,0
4,1990-11-18,2017-12-29,1,27,3,18,4,2019-11-03,2018-07-19,0


In [6]:
# Time variable
# Datum Rezidiv is the last date the patient was seen alive (last clinical contact).
# It is used as the censoring date for patients who did not die.
# Censoring is non-informative: the date reflects last follow-up, not recurrence status.
reduc_df["time"] = np.where(
    reduc_df["Tod Datum"].notna(),
    (reduc_df["Tod Datum"] - reduc_df["OPDate"]).dt.days,
    np.where(
        reduc_df["Datum Rezidiv"].notna(),
        (reduc_df["Datum Rezidiv"] - reduc_df["OPDate"]).dt.days,
        np.nan
    )
)
reduc_df['months'] = reduc_df['time'] / 30.44  # Convert days to months

# Event indicator
reduc_df["event"] = np.where(reduc_df["Tod Datum"].notna(), 1, 0)

In [7]:
# Validate survival times
neg = (reduc_df['time'] < 0).sum()
zero = (reduc_df['time'] == 0).sum()
missing = reduc_df['time'].isna().sum()
print(f'Negative survival times : {neg}')
print(f'Zero survival times     : {zero}')
print(f'Missing survival times  : {missing}')
print(f'Min time: {reduc_df["time"].min():.1f} days | Max time: {reduc_df["time"].max():.1f} days')
assert neg == 0, 'ERROR: negative survival times detected — check OPDate / Tod Datum entries'
short = (reduc_df['time'] < 7).sum()
print(f'Patients with < 7 days survival: {short}')
reduc_df[reduc_df['time'] < 7][['OPDate', 'Tod Datum', 'time']].head(10)

Negative survival times : 0
Zero survival times     : 0
Missing survival times  : 0
Min time: 12.0 days | Max time: 4848.0 days
Patients with < 7 days survival: 0


,OPDate,Tod Datum,time


In [8]:
reduc_df["Age_calc"] = (reduc_df["OPDate"] - reduc_df["Bday"]).dt.days / 365.25

reduc_df["Age_final"] = reduc_df["Age"]
reduc_df.loc[reduc_df["Age_final"].isna(), "Age_final"] = reduc_df["Age_calc"]

reduc_df["Sex"] = reduc_df["Sex"]-1

In [9]:
# Cross-validate reported Age against calculated age
mask = reduc_df['Age_calc'].notna() & reduc_df['Age'].notna()
age_diff = (reduc_df.loc[mask, 'Age'] - reduc_df.loc[mask, 'Age_calc']).abs()
suspicious = age_diff[age_diff > 2]
print(f'Patients with |Age - Age_calc| > 2 years: {len(suspicious)}')
if len(suspicious):
    display(reduc_df.loc[suspicious.index, ['Bday', 'OPDate', 'Age', 'Age_calc']]
            .assign(diff=age_diff[suspicious.index].round(1)))
else:
    print('All ages consistent with date of birth and operation date.')

Patients with |Age - Age_calc| > 2 years: 0
All ages consistent with date of birth and operation date.


In [10]:
reduc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285 entries, 0 to 284
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Bday           285 non-null    datetime64[ns]
 1   OPDate         285 non-null    datetime64[ns]
 2   Sex            285 non-null    int64         
 3   Age            285 non-null    int64         
 4   Tumor          285 non-null    int64         
 5   sPCI           285 non-null    int64         
 6   pPCI           285 non-null    int64         
 7   Tod Datum      131 non-null    datetime64[ns]
 8   Datum Rezidiv  263 non-null    datetime64[ns]
 9   CC             285 non-null    int64         
 10  time           285 non-null    float64       
 11  months         285 non-null    float64       
 12  event          285 non-null    int64         
 13  Age_calc       285 non-null    float64       
 14  Age_final      285 non-null    int64         
dtypes: datetime64[ns](4), f

In [11]:
# Keep tumour types with n >= 10, then explicitly exclude type 8
# (type 8 has exactly n=10 but is excluded for consistency with notebooks 02 and 04)
tumor_counts = reduc_df['Tumor'].value_counts()
tumors_to_keep = tumor_counts[tumor_counts >= 10].index
tumors_to_keep = tumors_to_keep[tumors_to_keep != 8]
reduc_df = reduc_df[reduc_df['Tumor'].isin(tumors_to_keep)].copy()
print(f'N after tumour filter: {len(reduc_df)}')
reduc_df['Tumor'].value_counts()

N after tumour filter: 254


Tumor
1    99
4    48
6    31
5    23
3    22
2    18
7    13
Name: count, dtype: int64

In [12]:
reduc_df = reduc_df[(reduc_df['CC']==0) | (reduc_df['CC']==1)].reset_index(drop=True)  # keep only CC0 and CC1
reduc_df["CC"].value_counts()

CC
0    170
1     83
Name: count, dtype: int64

In [13]:
tumor_dummies = pd.get_dummies(reduc_df, columns=["Tumor"], drop_first=True)
tumor_dummies.columns
reduc_df = pd.concat([reduc_df["Tumor"], tumor_dummies], axis=1)
reduc_df.head()

,Tumor,Bday,OPDate,Sex,Age,sPCI,pPCI,Tod Datum,Datum Rezidiv,CC,...,months,event,Age_calc,Age_final,Tumor_2,Tumor_3,Tumor_4,Tumor_5,Tumor_6,Tumor_7
0,1,1991-02-25,2008-03-10,1,17,26,22,2008-08-16,2008-08-12,1,...,5.223390,1,17.037645,17,False,False,False,False,False,False
1,1,1993-07-17,2015-07-14,0,21,5,3,2018-05-06,2016-12-22,0,...,33.738502,1,21.990418,21,False,False,False,False,False,False
2,4,1995-06-08,2018-11-23,1,23,15,6,NaT,2022-06-22,1,...,42.936925,0,23.460643,23,False,False,True,False,False,False
3,3,1990-11-18,2017-12-29,0,27,18,4,2019-11-03,2018-07-19,0,...,22.141919,1,27.112936,27,False,True,False,False,False,False
4,1,1984-05-14,2013-07-30,0,29,14,9,2015-02-25,2013-12-05,1,...,18.889619,1,29.210130,29,False,False,False,False,False,False


In [14]:
reduc_df.to_csv(f"{DATA_PATH}/GPT_processed_survival_data.csv", index=False)

In [15]:
reduc_df.shape

(253, 21)

In [16]:
reduc_df.columns

Index(['Tumor', 'Bday', 'OPDate', 'Sex', 'Age', 'sPCI', 'pPCI', 'Tod Datum',
       'Datum Rezidiv', 'CC', 'time', 'months', 'event', 'Age_calc',
       'Age_final', 'Tumor_2', 'Tumor_3', 'Tumor_4', 'Tumor_5', 'Tumor_6',
       'Tumor_7'],
      dtype='object')